# 14 — ChemBERTa-77M-MTR Pretrained SMILES Embeddings

Uses `deepchem/ChemBERTa-77M-MTR` — a RoBERTa model pretrained on 77M PubChem SMILES
with a **Multi-Task Regression (MTR)** objective (predicting molecular properties),
compared to notebook 13's `ChemBERTa-zinc-base-v1` which uses Masked Language Modeling (MLM).

**Hypothesis**: MTR pretraining on property prediction tasks (ESOL, FreeSolv, etc.) produces
representations more directly useful for activity regression than generic MLM.

**Strategy**: Frozen CLS-token embeddings (768-dim) + Morgan FP (2048) -> LGBM.
Same setup as notebook 13 for a clean head-to-head comparison.

**Runtime**: ~30–60 min (CPU inference for 4,652 SMILES + LGBM CV).

In [1]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import lightgbm as lgb
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

from pxr.data import load_train, load_test
from pxr.chem import bemis_murcko
from pxr.featurize import morgan, impute
from pxr.eval import scaffold_kfold_indices, compute_metrics, rae as rae_fn
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

plt.rcParams.update({"figure.dpi": 120})
DEVICE = 'cpu'
MODEL_NAME = 'deepchem/ChemBERTa-77M-MTR'
print(f"torch {torch.__version__}  |  device: {DEVICE}")
print(f"Model: {MODEL_NAME}")

torch 2.11.0+cpu  |  device: cpu
Model: deepchem/ChemBERTa-77M-MTR


In [2]:
# ── 2. Load data ──────────────────────────────────────────────────────────────
train = load_train()
te    = load_test()

smiles_tr = train['smiles'].tolist()
smiles_te = te['smiles'].tolist()
y_tr      = train['pec50'].values

scaffolds = train['smiles'].map(bemis_murcko).tolist()
splits    = scaffold_kfold_indices(scaffolds, n_splits=5, seed=42)

print(f"Train: {len(smiles_tr):,}  |  Test: {len(smiles_te):,}")

Train: 4,139  |  Test: 513


In [3]:
# ── 3. Load ChemBERTa-77M-MTR and extract embeddings ─────────────────────────
cache_path_tr = DATA_PROCESSED / 'chemberta_mtr_train_emb.npy'
cache_path_te = DATA_PROCESSED / 'chemberta_mtr_test_emb.npy'

if cache_path_tr.exists() and cache_path_te.exists():
    print("Loading cached ChemBERTa-77M-MTR embeddings ...")
    emb_tr = np.load(cache_path_tr)
    emb_te = np.load(cache_path_te)
else:
    print(f"Loading model from HuggingFace: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model     = AutoModel.from_pretrained(MODEL_NAME)
    model.eval()

    def encode_smiles(smiles_list: list, batch_size: int = 32) -> np.ndarray:
        all_embs = []
        for i in tqdm(range(0, len(smiles_list), batch_size), desc='Encoding'):
            batch = smiles_list[i : i + batch_size]
            enc   = tokenizer(batch, padding=True, truncation=True,
                               max_length=128, return_tensors='pt')
            with torch.no_grad():
                out = model(**enc)
            cls = out.last_hidden_state[:, 0, :].numpy()
            all_embs.append(cls)
        return np.vstack(all_embs)

    print("Encoding training set ...")
    emb_tr = encode_smiles(smiles_tr)
    print("Encoding test set ...")
    emb_te = encode_smiles(smiles_te)

    np.save(cache_path_tr, emb_tr)
    np.save(cache_path_te, emb_te)
    print("Cached to data/processed/")

print(f"Embeddings: train {emb_tr.shape}  test {emb_te.shape}")

Loading model from HuggingFace: deepchem/ChemBERTa-77M-MTR


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

Encoding training set ...


Encoding:   0%|          | 0/130 [00:00<?, ?it/s]

Encoding test set ...


Encoding:   0%|          | 0/17 [00:00<?, ?it/s]

Cached to data/processed/
Embeddings: train (4139, 384)  test (513, 384)


In [4]:
# ── 4. Build feature matrix: MolFormer + Morgan FP ────────────────────────────
X_morgan_tr = morgan(smiles_tr).astype(np.float32)
X_morgan_te = morgan(smiles_te).astype(np.float32)

X_tr = np.hstack([emb_tr, X_morgan_tr])   # (N, 768 + 2048)
X_te = np.hstack([emb_te, X_morgan_te])
print(f"Combined feature matrix: {X_tr.shape}")

Combined feature matrix: (4139, 2432)


In [5]:
# ── 5. Scaffold 5-fold CV with LGBM ──────────────────────────────────────────
lgbm_params = dict(
    n_estimators=1000, num_leaves=64, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    min_child_samples=10, n_jobs=4, verbose=-1,
)

oof_preds = np.full(len(y_tr), np.nan)
fold_metrics = []

for fold, (tr_idx, va_idx) in enumerate(splits):
    m = lgb.LGBMRegressor(**lgbm_params)
    m.fit(X_tr[tr_idx], y_tr[tr_idx])
    oof_preds[va_idx] = m.predict(X_tr[va_idx])
    met = compute_metrics(y_tr[va_idx], oof_preds[va_idx])
    met['fold'] = fold
    fold_metrics.append(met)
    print(f"  Fold {fold+1}: RAE={met['RAE']:.4f}  Spearman={met['Spearman']:.4f}")

oof_rae = rae_fn(y_tr, oof_preds)
cv_df   = pd.DataFrame(fold_metrics)
print(f"\nOOF RAE (global): {oof_rae:.4f}")
print(f"Mean fold RAE:    {cv_df['RAE'].mean():.4f} +/- {cv_df['RAE'].std():.4f}")
print()
print("== Comparison ==")
print(f"  LGBM_base (Morgan + RDKit only): ~0.575")
print(f"  LGBM_aug (+ null feature):       0.5582")
print(f"  ChemBERTa-zinc-MLM (nb 13):      0.6782")
print(f"  ChemBERTa-PubChem-MTR (this nb): {oof_rae:.4f}")

  Fold 1: RAE=0.5290  Spearman=0.7521


  Fold 2: RAE=0.6099  Spearman=0.6782


  Fold 3: RAE=0.6295  Spearman=0.6773


  Fold 4: RAE=0.5866  Spearman=0.6906


  Fold 5: RAE=0.6676  Spearman=0.6464

OOF RAE (global): 0.5993
Mean fold RAE:    0.6045 +/- 0.0516

== Comparison ==
  LGBM_base (Morgan + RDKit only): ~0.575
  LGBM_aug (+ null feature):       0.5582
  ChemBERTa-zinc-MLM (nb 13):      0.6782
  ChemBERTa-PubChem-MTR (this nb): 0.5993


In [6]:
# ── 6. Train final model + predict test ──────────────────────────────────────
final_model = lgb.LGBMRegressor(**lgbm_params)
final_model.fit(X_tr, y_tr)
mtr_preds = final_model.predict(X_te)
mtr_preds = np.clip(mtr_preds, y_tr.min() - 0.5, y_tr.max() + 0.5)

# Blend with Chemprop using inverse-RAE weights
for fname in ['10_expanded_multitask.csv', '08_chemprop_cv_blend.csv']:
    cp_path = SUBMISSIONS / fname
    if cp_path.exists():
        cp_preds = pd.read_csv(cp_path).set_index('Molecule Name').loc[te['name'].values, 'pEC50'].values
        break

chemprop_rae  = 0.5736
w_mtr = (1/oof_rae) / (1/oof_rae + 1/chemprop_rae)
final_preds = w_mtr * mtr_preds + (1 - w_mtr) * cp_preds
final_preds = np.clip(final_preds, y_tr.min() - 0.5, y_tr.max() + 0.5)

print(f"ChemBERTa-MTR test preds: {mtr_preds.min():.2f}–{mtr_preds.max():.2f}")
print(f"ChemBERTa-MTR weight: {w_mtr:.3f}  (RAE {oof_rae:.4f} vs Chemprop {chemprop_rae:.4f})")

ChemBERTa-MTR test preds: 2.83–5.83
ChemBERTa-MTR weight: 0.489  (RAE 0.5993 vs Chemprop 0.5736)


In [7]:
# ── 7. Save submission ────────────────────────────────────────────────────────
sub = pd.DataFrame({'Molecule Name': te['name'].values,
                    'SMILES':        te['smiles'].values,
                    'pEC50':         final_preds})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out = SUBMISSIONS / '14_chemberta_mtr.csv'
sub.to_csv(out, index=False)

np.save(DATA_PROCESSED / 'oof_chemberta_mtr.npy', oof_preds)
np.save(DATA_PROCESSED / 'te_chemberta_mtr.npy',  mtr_preds)

print(f"Saved: {out}")
print(f"ChemBERTa-MTR OOF RAE: {oof_rae:.4f}  |  blend weight: {w_mtr:.3f}")
print(sub['pEC50'].describe().round(3))

Saved: D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\14_chemberta_mtr.csv
ChemBERTa-MTR OOF RAE: 0.5993  |  blend weight: 0.489
count    513.000
mean       4.780
std        0.530
min        2.684
25%        4.474
50%        4.881
75%        5.184
max        5.716
Name: pEC50, dtype: float64
